# Analise de Correlacao — Wine Dataset

Exploracao dos componentes quimicos do vinho e suas correlacoes com a classe de cultivo.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

## 1. Carregamento dos dados

In [ ]:
df = pd.read_csv('../data/raw.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

## 2. Distribuicao das classes

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
df['cultivar'].value_counts().plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452','#55A868'])
ax.set_title('Distribuicao das Classes de Cultivo')
ax.set_xlabel('Cultivar')
ax.set_ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Matriz de correlacao entre features

In [ ]:
FEATURES = [
    'alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
    'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins',
    'color_intensity', 'hue', 'od280_od315', 'proline',
]

corr = df[FEATURES].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax
)
ax.set_title('Correlacao entre Componentes Quimicos do Vinho', fontsize=13)
plt.tight_layout()
plt.savefig('../data/correlacao_heatmap.png', dpi=120)
plt.show()

## 4. Features mais correlacionadas com cada cultivar

In [ ]:
# Codifica cultivar numericamente para calcular correlacao ponto-biserial
df_enc = df.copy()
df_enc['target_num'] = df_enc['target']

corr_target = df_enc[FEATURES + ['target_num']].corr()['target_num'].drop('target_num')
corr_target = corr_target.abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
corr_target.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Correlacao Absoluta de cada Feature com a Classe (target)')
ax.set_xlabel('|correlacao|')
plt.tight_layout()
plt.show()

print('Top 5 features mais correlacionadas com o cultivar:')
print(corr_target.head())

## 5. Distribuicao das top features por cultivar

In [ ]:
top_features = corr_target.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, feat in zip(axes.flat, top_features):
    for cultivar, grp in df.groupby('cultivar'):
        grp[feat].plot(kind='kde', ax=ax, label=cultivar)
    ax.set_title(feat)
    ax.legend()

fig.suptitle('Distribuicao das Top Features por Cultivar', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Pares mais correlacionados entre si (multicolinearidade)

Pares com |correlacao| > 0.7 merecem atencao: podem ser redundantes para modelos lineares.

In [ ]:
corr_pairs = (
    corr.where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['feature_a', 'feature_b', 'correlacao']
corr_pairs['abs'] = corr_pairs['correlacao'].abs()

high_corr = corr_pairs[corr_pairs['abs'] > 0.7].sort_values('abs', ascending=False)
print(f'Pares com |correlacao| > 0.7:')
print(high_corr[['feature_a', 'feature_b', 'correlacao']].to_string(index=False))